# Custom Environment Example using isopro

This notebook demonstrates how to create a custom environment using the `isopro` package with Claude or Hugging Face models.

In [ ]:
import isopro
from isopro import CustomEnvironment, AI_Agent
import anthropic
from transformers import pipeline
import os

## Creating a Custom Environment with Claude

In [ ]:
class ClaudeAgent(AI_Agent):
    def __init__(self, name):
        super().__init__(name)
        self.client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    def run(self, input_data):
        response = self.client.messages.create(
            model="claude-3-opus-20240229",
            max_tokens=100,
            messages=[{"role": "user", "content": input_data['text']}]
        )
        return response.content[0].text

class ClaudeEnvironment(CustomEnvironment):
    def __init__(self):
        super().__init__()
        self.agent = ClaudeAgent("Claude Agent")
        self.add_agent(self.agent)

    def step(self, action):
        response = self.agent.run({"text": action})
        return response, 0, False, {}

# Create and use the Claude environment
claude_env = ClaudeEnvironment()
response, _, _, _ = claude_env.step("Tell me a joke about AI")
print("Claude's response:", response)

## Creating a Custom Environment with Hugging Face

In [ ]:
class HuggingFaceAgent(AI_Agent):
    def __init__(self, name):
        super().__init__(name)
        self.model = pipeline("text-generation", model="gpt2")

    def run(self, input_data):
        response = self.model(input_data['text'], max_length=50, num_return_sequences=1)
        return response[0]['generated_text']

class HuggingFaceEnvironment(CustomEnvironment):
    def __init__(self):
        super().__init__()
        self.agent = HuggingFaceAgent("Hugging Face Agent")
        self.add_agent(self.agent)

    def step(self, action):
        response = self.agent.run({"text": action})
        return response, 0, False, {}

# Create and use the Hugging Face environment
hf_env = HuggingFaceEnvironment()
response, _, _, _ = hf_env.step("The future of AI is")
print("Hugging Face's response:", response)